In [1]:
'''
Objective: This code initializes required libraries for the Carla simulator to add a vehicle, and attach LiDAR and depth sensors to it. The libraries facilitate the simulation and visualization of the vehicle and sensor data.

Libraries Overview:
    - carla: Provides the interface to interact with the Carla simulator, allowing vehicle and sensor manipulation.
    - math: Supplies mathematical functions (e.g., for positioning, rotations).
    - random: Generates random numbers, useful for randomizing vehicle or sensor placement.
    - time: Provides time control (e.g., for pauses and timing).
    - numpy (np): Enables numerical operations on arrays, important for sensor data processing.
    - cv2: Handles image processing and display (for camera sensor data).
    - matplotlib and cm: Used for color mapping and data visualization (e.g., for depth sensor).
'''
import carla 
import math
import random
import time
import numpy as np
import cv2
from matplotlib import cm
import matplotlib

In [ ]:
'''
Objective: Establish a connection to the Carla simulator and access the simulation world.

1. carla.Client('localhost', 2000):
    - Purpose: Creates a client to connect to the Carla simulator running on localhost at port 2000.
    - Usage: Acts as the main interface to communicate with the Carla server.

2. client.get_world():
    - Purpose: Retrieves the current simulation environment or "world" in Carla.
    - Usage: Provides access to all elements in the simulation, such as vehicles, sensors, and actors, allowing manipulation within the world.
'''
client = carla.Client('localhost', 2000)
world = client.get_world()

In [3]:
'''
Objective: Retrieve available spawn points within the Carla simulation world.

1.world.get_map():
    - Purpose: Fetches the current map of the simulation world.
    - Usage: Provides information related to the simulation environment's layout (e.g., roads, intersections).

2.get_spawn_points():
    - Purpose: Returns a list of predefined spawn points (positions and orientations) on the map where vehicles can be placed.
    - Usage: Used to randomly or specifically select a location to spawn vehicles.
'''
spawn_points = world.get_map().get_spawn_points()

In [4]:
'''
Objective: Access the blueprint library to retrieve vehicle and sensor definitions.

1. world.get_blueprint_library():
    - Purpose: Fetches the blueprint library, which contains definitions for various actors, including vehicles, sensors, and props.
    - Usage: Enables selection and instantiation of specific vehicle or sensor types within the simulation, allowing customization and flexibility in vehicle configurations.
'''
bp_lib = world.get_blueprint_library()

In [5]:
'''
Objective: Spawn a specific vehicle in the Carla simulation.

1. `bp_lib.find('vehicle.tesla.cybertruck')`:
   - Purpose: Locates the blueprint for the Tesla Cybertruck within the blueprint library.
   - Usage: Retrieves the vehicle's specifications to allow for instantiation in the simulation.

2. `world.try_spawn_actor(vehicle_bp, spawn_points[0])`:
   - Purpose: Attempts to spawn an actor (the Tesla Cybertruck) at the first available spawn point.
   - Usage: Creates an instance of the vehicle in the simulation; if spawning is successful, it returns the vehicle object, allowing further interaction.
'''

vehicle_bp = bp_lib.find('vehicle.tesla.cybertruck')
vehicle = world.try_spawn_actor(vehicle_bp, spawn_points[0])

In [6]:
vehicle.set_autopilot(True)

In [7]:
'''
Objective: Define the initial camera position for the simulation.

1. `CAMERA_POS_Z = 3`:
   - Purpose: Sets the Z-axis position of the camera above the vehicle.
   - Usage: Specifies how high the camera will be placed relative to the vehicle, enhancing the field of view.

2. `CAMERA_POS_X = 0`:
   - Purpose: Sets the X-axis position of the camera.
   - Usage: Defines the lateral position of the camera relative to the vehicle, allowing for adjustments in viewpoint.

3. `camera_init_trans = carla.Transform(carla.Location(z = CAMERA_POS_Z, x = CAMERA_POS_X))`:
   - Purpose: Creates a transformation object representing the camera's initial position.
   - Usage: Combines the defined X and Z positions into a transform that can be used when attaching the camera to the vehicle.
'''
CAMERA_POS_Z = 3
CAMERA_POS_X = 0
camera_init_trans = carla.Transform(carla.Location(z = CAMERA_POS_Z, x = CAMERA_POS_X))

In [8]:
'''
Objective: Spawn an RGB camera sensor and attach it to the vehicle.

1. `camera_bp = bp_lib.find('sensor.camera.rgb')`:
   - Purpose: Locates the blueprint for the RGB camera within the blueprint library.
   - Usage: Retrieves the specifications needed to instantiate the camera in the simulation.

2. `camera = world.spawn_actor(camera_bp, camera_init_trans, attach_to=vehicle)`:
   - Purpose: Spawns the RGB camera actor at the specified initial transform and attaches it to the vehicle.
   - Usage: Ensures that the camera moves with the vehicle, allowing for real-time image capture during the simulation.
'''
camera_bp = bp_lib.find('sensor.camera.rgb')
camera = world.spawn_actor(camera_bp, camera_init_trans, attach_to=vehicle)

In [9]:
'''
Objective: Spawn a GNSS sensor and attach it to the vehicle.

1. `gnss_bp = bp_lib.find('sensor.other.gnss')`:
    - Purpose: Locates the blueprint for the GNSS sensor within the blueprint library.
    - Usage: Retrieves the necessary specifications to instantiate the GNSS sensor in the simulation. The GNSS sensor provides real-time geographical data such as latitude, longitude, and altitude.

2. `gnss_sensor = world.spawn_actor(gnss_bp, camera_init_trans, attach_to=vehicle)`:
    - Purpose: Spawns the GNSS sensor actor at the specified initial transform and attaches it to the vehicle.
    - Usage: Ensures that the GNSS sensor remains attached to the vehicle, continuously providing accurate location data during the simulation.
'''
gnss_bp = bp_lib.find('sensor.other.gnss')
gnss_sensor = world.spawn_actor(gnss_bp, camera_init_trans, attach_to=vehicle)

In [10]:
'''
Objective: Spawn an IMU (Inertial Measurement Unit) sensor and attach it to the vehicle.

1. `imu_bp = bp_lib.find('sensor.other.imu')`:
    - Purpose: Locates the blueprint for the IMU sensor within the blueprint library.
    - Usage: Retrieves the necessary specifications to instantiate the IMU sensor in the simulation. The IMU sensor provides measurements of the vehicle’s acceleration, angular velocity, and orientation.

2. `imu_sensor = world.spawn_actor(imu_bp, camera_init_trans, attach_to=vehicle)`:
    - Purpose: Spawns the IMU sensor actor at the specified initial transform and attaches it to the vehicle.
    - Usage: Ensures that the IMU sensor remains attached to the vehicle, providing continuous measurements of acceleration and rotation during the simulation.
'''
imu_bp = bp_lib.find('sensor.other.imu')
imu_sensor = world.spawn_actor(imu_bp, camera_init_trans, attach_to = vehicle)

In [11]:
'''
Objective: Process and store incoming camera images.

1. `def camera_callback(image, data_dict):`
   - Purpose: Defines a callback function that processes raw image data from the camera.
   - Parameters:
     - `image`: The raw image data received from the camera sensor.
     - `data_dict`: A dictionary where the processed image will be stored.

2. Inside the function:
   - `data_dict['image'] = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))`:
     - Purpose: Copies the raw image data and reshapes it into a 4-channel image (including an alpha channel).
     - `np.copy(image.raw_data)`: Creates a copy of the raw image data to prevent modifications to the original data.
     - `np.reshape(..., (image.height, image.width, 4))`: Reshapes the data into a format suitable for further processing, where `image.height` and `image.width` correspond to the image's dimensions.

3. Usage: This function allows real-time updates of the camera image data for visualization or analysis.

'''
def camera_callback(image, data_dict):
    data_dict['image'] = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))

In [12]:
'''
Objective: Define a callback function to process GNSS data and store the latitude and longitude in a dictionary.

1. def gnss_callback(data, data_dict)::
    - Purpose: Defines a callback function that gets triggered whenever the GNSS sensor provides new data.
    - Usage: The function takes two arguments:
        1. data: Contains the GNSS data (latitude, longitude, etc.).
        2. data_dict: A dictionary where the processed GNSS data (latitude and longitude) will be stored for later use.

2. data_dict['gnss'] = [data.latitude, data.longitude]:
    - Purpose: Updates the data_dict with the current latitude and longitude from the data object.
    - Usage: The latitude and longitude are stored in a list format as ['latitude', 'longitude'] under the 'gnss' key in the data_dict. This allows easy access to GNSS data throughout the program.

'''
def gnss_callback(data, data_dict):
    data_dict['gnss'] = [data.latitude, data.longitude]

In [13]:
'''
Objective: Define a callback function to process IMU data and store gyroscope, accelerometer, and compass readings in a dictionary.

1. def imu_callback(data, data_dict):
    - Purpose: Defines a callback function that is triggered whenever the IMU sensor provides new data.
    - Usage: The function takes two arguments:
        1. data: Contains the IMU data (gyroscope, accelerometer, compass).
        2. data_dict: A dictionary where the processed IMU data will be stored for further use.

2. data_dict['imu'] = { 'gyro': data.gyroscope, 'accel': data.accelerometer, 'compass': data.compass }:
    - Purpose: Updates the data_dict with the current readings from the IMU sensor.
    - Usage: The gyroscope, accelerometer, and compass data are organized in a dictionary format under the 'imu' key in data_dict. This allows for structured access to IMU sensor data, enabling easy retrieval of sensor readings.
'''

def imu_callback(data, data_dict):
    data_dict['imu'] = {
        'gyro' : data.gyroscope, 
        'accel' : data.accelerometer,
        'compass' : data.compass
    }



In [14]:
''' 
Objective: Retrieve the dimensions of the camera's image.

1. `image_w = camera_bp.get_attribute("image_size_x").as_int()`:
   - Purpose: Fetches the width of the camera's image in pixels.
   - Usage: Stores the image width, which can be used for processing or displaying the captured images.

2. `image_h = camera_bp.get_attribute("image_size_y").as_int()`:
   - Purpose: Fetches the height of the camera's image in pixels.
   - Usage: Stores the image height, ensuring that the dimensions are available for image processing and visualization tasks.

'''
image_w = camera_bp.get_attribute("image_size_x").as_int()
image_h = camera_bp.get_attribute("image_size_y").as_int()

In [15]:
'''
Objective: Initialize a dictionary to store sensor data, including image, GNSS, and IMU readings.

1. sensor_data = {...}:
    - Purpose: Creates a dictionary named sensor_data to hold various sensor data collected during simulation.
    - Usage: The dictionary contains three main keys: 'image', 'gnss', and 'imu', each designed to store specific types of sensor information.

2. 'image': np.zeros((image_h, image_w, 4)):
    - Purpose: Initializes the 'image' key with a NumPy array filled with zeros, representing an empty image.
    - Usage: The array dimensions are defined by image_h (height) and image_w (width), with a depth of 4 to accommodate RGBA (Red, Green, Blue, Alpha) channels.

3. 'gnss': [0.0]:
    - Purpose: Initializes the 'gnss' key with a list containing a single floating-point value (0.0).
    - Usage: This list is intended to store GNSS data (e.g., latitude and longitude). The initial value indicates that no GNSS data has been captured yet.

4. 'imu': {...}:
    - Purpose: Initializes the 'imu' key with a nested dictionary to store IMU sensor readings.
    - Usage: This nested dictionary contains:
        1. 'gyro': carla.Vector3D(): Initializes the gyroscope reading as a 3D vector using the CARLA API, representing the angular velocity around the X, Y, and Z axes.
        2. 'accel': carla.Vector3D(): Initializes the accelerometer reading as a 3D vector, representing acceleration in the three spatial dimensions.
        3. 'compass': 0: Initializes the compass reading with a value of 0, which can be updated with the actual compass data later.

'''

sensor_data = {'image': np.zeros((image_h, image_w, 4)),
               'gnss' : [0.0],
               'imu' : {
                   'gyro' : carla.Vector3D(),
                   'accel': carla.Vector3D(),
                   'compass' : 0
               }}

In [16]:
'''
Objective: Display real-time data from RGB camera, GNSS, and IMU sensors.

1. cv2.namedWindow('RGB Camera', cv2.WINDOW_AUTOSIZE):
    - Purpose: Creates a window named 'RGB Camera' for displaying the camera feed.
    - Usage: The window will automatically resize based on the displayed image.

2. cv2.imshow('RGB Camera', sensor_data['image']):
    - Purpose: Displays the current image stored in sensor_data['image'] in the 'RGB Camera' window.
    - Usage: Provides a real-time visual representation of the camera feed.

3. cv2.waitKey(1):
    - Purpose: Waits for a specified amount of time for a key event.
    - Usage: In this case, it waits for 1 millisecond to allow the window to refresh.

4. camera.listen(lambda image: camera_callback(image, sensor_data)):
    - Purpose: Sets up a listener for the camera that triggers the camera_callback function when new image data is available.
    - Usage: Ensures the camera data is processed and stored in the sensor_data dictionary.

5. gnss_sensor.listen(lambda event: gnss_callback(event, sensor_data)):
    - Purpose: Sets up a listener for the GNSS sensor to update the sensor_data with new location data.
    - Usage: Allows real-time GNSS updates for latitude and longitude.

6. imu_sensor.listen(lambda event: imu_callback(event, sensor_data)):
    - Purpose: Sets up a listener for the IMU sensor to update the sensor_data with new IMU readings.
    - Usage: Enables real-time updates for gyroscope, accelerometer, and compass data.

Text Formatting Parameters:
- font = cv2.FONT_HERSHEY_SIMPLEX: Sets the font style for text overlay.
- bottomleftCornerOfText = (10, 50): Defines the initial position for the text overlay.
- fontScale = 0.5: Sets the scale of the font.
- fontColor = (255, 255, 255): Specifies the font color (white).
- tickness = 2: Sets the thickness of the text.
- lineType = 2: Defines the line type for text rendering.

Main Loop:
- Purpose: Continuously updates and displays the sensor data on the image.
Usage:
Latitude and Longitude:
1. cv2.putText(sensor_data['image'], 'Lat: ' + str(sensor_data['gnss'][0]), (10,30), font, fontScale, fontColor, tickness, lineType): Displays the current latitude.
2. cv2.putText(sensor_data['image'], 'Long: ' + str(sensor_data['gnss'][1]), (10, 50), font, fontScale, fontColor, tickness, lineType): Displays the current longitude.
Acceleration:
1. accel = sensor_data['imu']['accel'] - carla.Vector3D(x=0, y=0, z=0.81): Adjusts the acceleration by subtracting gravity.
2. cv2.putText(sensor_data['image'], 'Accel: ' + str(accel.length()), (10, 70), font, fontScale, fontColor, tickness, lineType): Displays the adjusted acceleration.
Gyroscope Reading:
1. cv2.putText(sensor_data['image'], 'Gyro: ' + str(sensor_data['imu']['gyro'].length()), (10, 100), font, fontScale, fontColor, tickness, lineType): Displays the gyroscope data.
2. cv2.imshow('RGB Camera', sensor_data['image']):

- Purpose: Updates the displayed image in the 'RGB Camera' window with the latest sensor data.
- Usage: Ensures the window reflects the most recent image data.

Exit Condition:
if cv2.waitKey(1) == ord('q')::
- Purpose: Checks if the 'q' key is pressed.
- Usage: If 'q' is pressed, the loop breaks, terminating the program.
'''

cv2.namedWindow('RGB Camera', cv2.WINDOW_AUTOSIZE)
cv2.imshow('RGB Camera', sensor_data['image'])
cv2.waitKey(1)

camera.listen(lambda image: camera_callback(image, sensor_data))
gnss_sensor.listen(lambda event: gnss_callback(event, sensor_data))
imu_sensor.listen(lambda event: imu_callback(event, sensor_data))

font = cv2.FONT_HERSHEY_SIMPLEX
bottomleftCornerOfText = (10, 50)
fontScale = 0.5
fontColor = (255, 255, 255)
tickness = 2
lineType = 2


while True:
    cv2.putText(sensor_data['image'], 'Lat: ' + str(sensor_data['gnss'][0]),
                (10,30),
                font,
                fontScale,
                fontColor,
                tickness,
                lineType)

    cv2.putText(sensor_data['image'], 'Long: ' + str(sensor_data['gnss'][1]),
    (10, 50),
    font,
    fontScale,
    fontColor,
    tickness,
    lineType)

    accel = sensor_data['imu']['accel'] - carla.Vector3D(x=0, y=0, z=0.81)

    cv2.putText(sensor_data['image'], 'Accel: ' + str(accel.length()),
    (10, 70),
    font,
    fontScale,
    fontColor,
    tickness,
    lineType)

    cv2.putText(sensor_data['image'], 'Gyro: ' + str(sensor_data['imu']['gyro'].length()),
    (10, 100),
    font,
    fontScale,
    fontColor,
    tickness,
    lineType)

   
    cv2.imshow('RGB Camera', sensor_data['image'])
    if cv2.waitKey(1) == ord('q'):
        break

cv2.destroyAllWindows()
camera.stop()
camera.destroy()
imu_sensor.stop()
gnss_sensor.destroy()
for actor in world.get_actors().filter('*vehicle*'):
    actor.destroy()
for sensor in world.get_actors().filter('*sensor*'):
    sensor.destroy()